## Gold Market Price

> **Scope Note:** This is an **Gold table**. It was added to make market price history directly queryable in Gold.

### Gold_Market_Price

Contains the complete market price history from:

`silver.market_price`

The Silver table contains **205 rows** with the following schema:

- `ticker`
- `bar_timestamp`
- `open`
- `high`
- `low`
- `close`
- `volume`

### Enrichment

The market price data is **LEFT JOINed** with portfolio company information using:

`market_price.ticker = portfolio_company.benchmark_ticker`

Additional fields:

- `company_id`
- `company_name`
- `fund_id`

### Join Behavior

A ticker matching a portfolio company's benchmark ticker receives the corresponding company and fund information.

A ticker with **no matching company** is still retained, with:

- `company_id = NULL`
- `company_name = NULL`
- `fund_id = NULL`

### Result

Every price bar ingested from the Bronze source remains queryable in Gold, including:

- Prices associated with project holdings
- Broader market-reference tickers
- Tickers without a matching portfolio company

### Gold Table Created

| Gold Table | Source | Logic |
|---|---|---|
| `Gold_Market_Price` | `silver.market_price` + portfolio company reference | Full price history + LEFT enrichment |


In [0]:
%run ./silver_common

In [0]:
from pyspark.sql import functions as F

## `Gold_Market_Price`

Left join keeps every price bar even when no portfolio company uses that ticker - `company_id`/`company_name`/`fund_id` are NULL in that case, not the row dropped. A ticker mapped to more than one company (not expected in this data, but not structurally impossible) would fan out one price row per match - the sanity check below confirms whether that happens.

In [0]:
market_price_df = spark.table(silver_table("market_price"))
portfolio_company_df = spark.table(silver_table("portfolio_company"))

# One row per ticker->company mapping (distinct, in case of any duplicate
# benchmark_ticker across companies - checked below).
ticker_to_company_df = (
    portfolio_company_df
    .select("company_id", "company_name", "fund_id", "benchmark_ticker")
    .filter(F.col("benchmark_ticker").isNotNull())
)

dup_ticker_check = (
    ticker_to_company_df
    .groupBy("benchmark_ticker")
    .agg(F.count("*").alias("company_count"))
    .filter(F.col("company_count") > 1)
)
dup_ticker_count = dup_ticker_check.count()
if dup_ticker_count > 0:
    print(f"WARNING: {dup_ticker_count} ticker(s) map to more than one company - price rows for these will fan out.")
    dup_ticker_check.show(truncate=False)

market_price_gold_df = (
    market_price_df
    .join(
        ticker_to_company_df,
        market_price_df["ticker"] == ticker_to_company_df["benchmark_ticker"],
        how="left"
    )
    .drop("benchmark_ticker")
    .withColumn("gold_loaded_at", F.current_timestamp())
)

write_gold(market_price_gold_df, "market_price")
print(f"Gold_Market_Price row count: {market_price_gold_df.count()}")

matched_count = market_price_gold_df.filter(F.col("company_id").isNotNull()).count()
unmatched_count = market_price_gold_df.filter(F.col("company_id").isNull()).count()
print(f"Price bars matched to a portfolio company: {matched_count}")
print(f"Price bars with no matching company (broader reference data): {unmatched_count}")

market_price_gold_df.orderBy("ticker", "bar_timestamp").show(30, truncate=False)

+----------------+-------------+
|benchmark_ticker|company_count|
+----------------+-------------+
|OR.PA           |2            |
|ULVR.L          |2            |
|NESN.SW         |2            |
|HSBA.L          |2            |
|BP.L            |2            |
|GOOGL           |4            |
+----------------+-------------+

Gold_Market_Price row count: 271
Price bars matched to a portfolio company: 188
Price bars with no matching company (broader reference data): 83
+-------+-------------------+--------+-------+-------+--------+--------+--------+--------------------------+----------+--------------------------+--------+--------------------------+
|ticker |bar_timestamp      |open    |high   |low    |close   |volume  |_run_id |_ingested_at              |company_id|company_name              |fund_id |gold_loaded_at            |
+-------+-------------------+--------+-------+-------+--------+--------+--------+--------------------------+----------+--------------------------+--------+---

### Summary

`Gold_Market_Price` is now queryable directly - by raw `ticker`, or by `fund_id`/`company_id` for the subset of tickers tied to an actual holding. 

In [0]:
cnt = spark.table(gold_table("market_price")).count()
print(f"Gold_Market_Price: {cnt} rows")

Gold_Market_Price: 271 rows
